# Lab 5: A2A for a publishing decision

An analyst wants to publish a short buy note on the ticker EXMP. Two facts decide it, and two other teams own them. The market-data team knows the latest quote, and the compliance team keeps the restricted list. In this lab each team's Agent runs behind its own Agent-to-Agent (A2A) server, and your orchestrator reaches them only over HTTP.

The orchestrator has no domain tools. `A2AClientToolProvider` gives it three client tools: one lists the peers it was told about, one fetches a peer's agent card, and one sends a peer a message. Everything the orchestrator learns about a peer comes from that peer's agent card and its replies.

## What you will do

1. Start two A2A servers in this notebook, and run the orchestrator against them.
2. Read the evidence: each agent card, the messages the orchestrator sent, and the conversations each server kept.
3. Send a follow-up with the same `context_id` and with a new one, and see how the server keeps them apart.

**Estimated time:** 20-25 minutes.

Every cell is ready to run. Run the notebook from the top and read the output of each cell.

![The publishing assistant sends an A2A message to a market data peer and a compliance peer, each running its own server with its own tools behind an ownership boundary.](images/topology.png)


## Why A2A

When every Agent lives in one Python process, an orchestrator can hold its specialists as objects and call them directly. That stops working once another team owns the specialist, deploys it on its own schedule, or builds it with another framework. A2A is a protocol for that boundary. A server publishes an agent card at `/.well-known/agent-card.json` with its name, description, and skills, and it accepts JSON-RPC `message/send` requests.

Nothing else crosses the boundary: no memory, no credentials, no process state. Each message carries its parts (text here) and, optionally, a `contextId` that names the conversation it belongs to. What a context holds is up to the server. In this lab each server builds a new Agent for each new context, through the `agent_factory` you pass to Strands' `A2AServer`.

The servers run on `127.0.0.1` in background threads of this notebook, so the lab needs only Amazon Bedrock and creates no AWS resources. In production each one would be its own service, and the caller would change only the URLs and add authentication.

## Setup

Run the `%pip` cell by itself first. If it installs or upgrades anything, restart the kernel. Then select the next cell and choose **Run > Run All Below**. Starting **Run All** above the install cell can stop after a package-triggered kernel restart.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import re
import socket
import sys
import threading
import time
import uuid
from contextlib import closing
from pathlib import Path

import boto3
import httpx
import uvicorn
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands_tools.a2a_client import A2AClientToolProvider

sys.path.insert(0, str(Path("..").resolve()))
from utils.live_panes import LivePanes, participant_text, show_conversations
from utils.validation import write_validation_artifact

The specialists run on Amazon Nova Pro and the orchestrator on Anthropic Claude Sonnet 5, both in the AWS Region of this notebook's session.

In [ ]:
region = boto3.Session().region_name
nova = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)

## The specialists

Each team owns one tool over a small fixture, and neither tool can answer the other team's question. `tool_runs` records every tool call, so you can check later which tools ran.

The two functions below are the `agent_factory` arguments of the servers. `A2AServer` calls its factory once at startup, with the placeholder context `__agent_card__`, only to read the name, description, and tools for the agent card. After that the server calls the factory the first time it sees a new `context_id`, and sends later messages with that ID to the same Agent. `built` records every Agent a factory creates, with its context.

Every Agent sets `callback_handler=None`: these Agents run inside this kernel, and their default handler would print their replies into whichever cell happens to be running.

In [ ]:
# Synthetic market and compliance data, and logs for inspection.
QUOTES = {"EXMP": "EXMP last traded at 41.20 USD, up 1.8% on the session."}
RESTRICTED = {"EXMP": "an open advisory deal blocks published research until it closes"}
tool_runs = []  # one entry per tool call, on any server
built = []  # (server, context_id, Agent) for every Agent a factory builds


# The specialists' tools. Each one records its call in tool_runs.
@tool
def latest_quote(ticker: str) -> str:
    """Return the latest quote for a ticker from the market-data feed."""
    tool_runs.append({"tool": "latest_quote", "ticker": ticker})
    return QUOTES.get(ticker.upper(), f"No quote for {ticker}.")


@tool
def restricted_status(ticker: str) -> str:
    """Return whether a ticker is on the compliance restricted list, and why."""
    tool_runs.append({"tool": "restricted_status", "ticker": ticker})
    reason = RESTRICTED.get(ticker.upper())
    return (
        f"{ticker.upper()} is on the restricted list: {reason}." if reason else f"{ticker.upper()} is not restricted."
    )


# Agent factories. A2AServer calls one for each new context_id.
def market_data_specialist(context_id: str) -> Agent:
    agent = Agent(
        name="market_data_specialist",
        description="Reports the latest quote for a ticker. Owns market data only.",
        model=nova,
        system_prompt=(
            "You are the market-data specialist. Call latest_quote for the ticker you are asked about and "
            "report the quote in one sentence. You know nothing about compliance or restricted lists."
        ),
        tools=[latest_quote],
        callback_handler=None,
    )
    built.append(("market_data", context_id, agent))
    return agent


def compliance_specialist(context_id: str) -> Agent:
    agent = Agent(
        name="compliance_specialist",
        description="Reports whether a ticker is on the restricted list. Owns compliance only.",
        model=nova,
        system_prompt=(
            "You are the compliance specialist. When asked about a ticker, call restricted_status and report "
            "the status in one sentence. Answer questions about this conversation from its history. "
            "You have no market data."
        ),
        tools=[restricted_status],
        callback_handler=None,
    )
    built.append(("compliance", context_id, agent))
    return agent

## The servers

`A2AServer.serve()` blocks the cell until the server stops. `BackgroundA2AServer` instead runs the server's Starlette app on a `uvicorn.Server` in a daemon thread, on a free localhost port, so several peers can run while you keep working. The clean-up cell stops them.

`enable_a2a_compliant_streaming=True` selects the response format the A2A specification expects. Without it, the pinned Strands version warns and keeps an older format.

In [ ]:
# Server settings, and the notebook's own HTTP client.
HOST = "127.0.0.1"
CARD_CONTEXT = "__agent_card__"  # the placeholder context A2AServer uses to build the agent card
a2a_http = httpx.Client(timeout=120)  # this notebook's own HTTP client, for cards and direct messages


# A free local port for each server.
def free_port() -> int:
    with closing(socket.socket(socket.AF_INET, socket.SOCK_STREAM)) as probe:
        probe.bind((HOST, 0))
        return probe.getsockname()[1]


# A server wrapper that starts and stops one peer.
class BackgroundA2AServer:
    """Run one A2AServer in a daemon thread so the notebook stays interactive."""

    def __init__(self, agent_factory, port: int):
        self.port = port
        self.url = f"http://{HOST}:{port}"
        a2a_server = A2AServer(
            agent_factory=agent_factory, host=HOST, port=port, http_url=self.url, enable_a2a_compliant_streaming=True
        )
        self.name = a2a_server.name
        config = uvicorn.Config(a2a_server.to_starlette_app(), host=HOST, port=port, log_level="warning")
        self.server = uvicorn.Server(config)
        self.thread = threading.Thread(target=self.server.run, name=f"a2a-{self.name}", daemon=True)

    def start(self, timeout: float = 20.0) -> "BackgroundA2AServer":
        self.thread.start()
        deadline = time.monotonic() + timeout
        while not self.server.started:
            if time.monotonic() > deadline or not self.thread.is_alive():
                raise TimeoutError(f"{self.name} did not start on port {self.port}")
            time.sleep(0.05)
        print(f"{self.name} serving A2A at {self.url}")
        return self

    def stop(self) -> None:
        self.server.should_exit = True
        self.thread.join(timeout=15.0)
        assert not self.thread.is_alive(), f"{self.name} did not stop"
        print(f"{self.name} stopped")


# Start both peers, each on its own port.
servers = {
    "market_data": BackgroundA2AServer(market_data_specialist, port=free_port()).start(),
    "compliance": BackgroundA2AServer(compliance_specialist, port=free_port()).start(),
}
peer_urls = [server.url for server in servers.values()]

## What to look for

The orchestrator gets the two URLs and a task.

Look for:

- How many messages does the orchestrator send to each peer?
- How many Agents does each server build, not counting the agent card?
- Which server runs `restricted_status`?
- Does the orchestrator publish or block the note?


## Run the orchestrator

The cell builds the orchestrator with only the client tools, then loops over `orchestrator.stream_async(TASK)` and shows each event in a pane. Only the orchestrator gets a pane. The specialists sit behind HTTP, so their replies arrive as tool results, as they would from another team's service.

**What to look for:** `[tool: a2a_list_discovered_agents]` or `[tool: a2a_discover_agent]`, then one `[tool: a2a_send_message]` for each peer, then the decision.

In [ ]:
# The request: two questions, each owned by another team.
TASK = (
    "May we publish a buy note on EXMP? Ask the market-data specialist for the latest quote and the "
    "compliance specialist for the restricted-list status, then decide."
)

# The orchestrator reaches the peers only through A2A client tools.
provider = A2AClientToolProvider(known_agent_urls=peer_urls)
orchestrator = Agent(
    name="orchestrator",
    model=sonnet,
    system_prompt=(
        "You decide whether the research team may publish a buy note. You have no market or compliance data "
        "of your own. List the discovered agents, then send each specialist only its own question with "
        "a2a_send_message. Do not answer either question yourself, and do not retry a failed call. "
        "Reply with PUBLISH or BLOCK on the first line, then one sentence per specialist citing its answer, "
        "or naming the specialist that did not answer."
    ),
    tools=provider.tools,
    callback_handler=None,
)

# Run the orchestrator and stream it into its pane.
panes = LivePanes()
async for event in orchestrator.stream_async(TASK):
    panes.show(event)
result = event["result"]

## Inspect the evidence

None of this evidence is the orchestrator's prose. The agent cards are what each peer publishes about itself. The orchestrator's `a2a_send_message` calls show whom it asked and whether each call succeeded. The server-side records show which Agents the factories built and which tools ran.

Compare this output with the questions under **What to look for**.

In [ ]:
# The peers' agent cards, as any caller sees them.
cards = {url: a2a_http.get(f"{url}/.well-known/agent-card.json").json() for url in peer_urls}
names = {url: card["name"] for url, card in cards.items()}
for card in cards.values():
    skills = [skill["id"] for skill in card["skills"]]
    print(f"{card['name']}  {card['url']}  protocol {card['protocolVersion']}")
    print(f"  {card['description']}  skills: {skills}")

# The orchestrator's view: each a2a_send_message call, and the status of its tool result.
results = {
    block["toolResult"]["toolUseId"]: block["toolResult"]["content"][0]["text"]
    for message in orchestrator.messages
    for block in message["content"]
    if "toolResult" in block
}

sends = [
    block["toolUse"]
    for message in orchestrator.messages
    for block in message["content"]
    if block.get("toolUse", {}).get("name") == "a2a_send_message"
]

print("\nmessages the orchestrator sent")
for send in sends:
    peer = names.get(send["input"]["target_agent_url"].rstrip("/"), "unknown URL")
    status = re.search(r"""["']status["']: ["'](\w+)""", results[send["toolUseId"]]).group(1)
    print(f"  -> {peer} [{status}]: {send['input']['message_text'][:100]}")

# The servers' view: every Agent a factory built, and every tool call.
print("\nAgents built by the factories")
for server, context_id, agent in built:
    print(f"  {server:12} context {context_id[:14]:14}  messages held: {len(agent.messages)}")
print(f"\ntool runs: {[(run['tool'], run['ticker']) for run in tool_runs]}")

# Check the route, tool ownership, and one context per message.
called = {names.get(send["input"]["target_agent_url"].rstrip("/")) for send in sends}
assert called == set(names.values()), f"peers called: {called}"
assert not {"latest_quote", "restricted_status"} & set(orchestrator.tool_names), "the orchestrator owns a domain tool"
assert {run["tool"] for run in tool_runs} >= {"latest_quote", "restricted_status"}, "a specialist did not use its tool"
for key, server in servers.items():
    contexts = [c for s, c, _ in built if s == key and c != CARD_CONTEXT]
    sent = sum(send["input"]["target_agent_url"].rstrip("/") == server.url for send in sends)
    assert len(contexts) == sent, f"{key}: {len(contexts)} contexts for {sent} messages"

The last check holds because `A2AClientToolProvider` sends no `contextId`. The server then gives every message a new context, and each question reaches a fresh Agent with no history. That suits one-shot questions. A follow-up needs the caller to send the `contextId` of the conversation it means, which is what the next section shows.

## Look inside the peers

The orchestrator's pane stops at the A2A boundary: it shows each `a2a_send_message` call and the reply, not the work behind it. Another team's server would show that work only in its own logs or traces. Here the servers run in this kernel, and `built` holds every Agent their factories built, so you can read each one's conversation: the message that arrived over HTTP, the tool it called, the tool result, and the reply it sent back.

**What to look for:** each Agent holds one question and one answer, and the only tool it called is its own team's.

In [ ]:
show_conversations({f"{server} {context[:8]}": agent for server, context, agent in built if context != CARD_CONTEXT})

## Follow up with the same and a new `context_id`

These cells skip the orchestrator and talk to the compliance server directly with `send()`, which posts one JSON-RPC `message/send` request. The first cell asks about EXMP in a new context. The second sends one follow-up in the same context and one in a context nobody has used.

Look for:

- Does the follow-up in the same context know the ticker?
- Does the follow-up in a new context know it?
- How many Agents has the compliance factory built for the first context?


In [ ]:
# Send one A2A message directly, with an explicit context_id.
def send(url: str, text: str, context_id: str) -> str:
    """POST one A2A message/send request and return the text of the reply."""
    message = {
        "role": "user",
        "messageId": uuid.uuid4().hex,
        "contextId": context_id,
        "parts": [{"kind": "text", "text": text}],
    }
    request = {"jsonrpc": "2.0", "id": uuid.uuid4().hex, "method": "message/send", "params": {"message": message}}
    task = a2a_http.post(url, json=request).raise_for_status().json()["result"]
    reply = "".join(part.get("text", "") for artifact in task.get("artifacts", []) for part in artifact["parts"])
    return participant_text(reply).strip()


# A first message to the compliance peer, in a new context.
compliance_url = servers["compliance"].url
first_context = uuid.uuid4().hex
print(send(compliance_url, "Is EXMP on the restricted list?", first_context))

In [ ]:
same_context = first_context  # the conversation that asked about EXMP
new_context = uuid.uuid4().hex  # a conversation nobody has started

follow_up = "Which ticker did I ask you about earlier in this conversation? If there was no earlier question, say so."
same_reply = send(compliance_url, follow_up, same_context)
new_reply = send(compliance_url, follow_up, new_context)

### Verify

The check reads the servers' records, not the replies: which Agent each context reached, and what that Agent holds.

In [ ]:
first_agents = [agent for server, c, agent in built if server == "compliance" and c == first_context]
new_agents = [agent for server, c, agent in built if server == "compliance" and c == new_context]
print(f"same context: {' '.join(same_reply.split())[:160]}")
print(f"new context : {' '.join(new_reply.split())[:160]}")
print(f"\nAgents for the first context: {len(first_agents)}, holding {len(first_agents[0].messages)} messages")
print(f"Agents for the new context  : {len(new_agents)}, holding {len(new_agents[0].messages)} messages")
print(f"first message the new context's Agent saw: {new_agents[0].messages[0]['content'][0]['text'][:80]}")

assert len(first_agents) == 1, "the factory built a second Agent for a context it already had"
assert "EXMP" in first_agents[0].messages[0]["content"][0]["text"], "the first context lost its first question"
assert new_agents[0].messages[0]["content"][0]["text"] == follow_up, "the new context saw an earlier message"

The same `context_id` reached the same Agent, with the first question in its history. The new one reached a fresh Agent that had never seen EXMP. `agent_factory` is what makes that separation real: with the deprecated single `agent=` argument, every context shares one Agent instance, and the executor has to swap histories in and out of it.

A `context_id` is routing, not authentication. Nothing in the request said who was calling, and any caller that learns or guesses a context ID can read and extend that conversation. Authenticate callers (for example with OAuth bearer tokens or IAM SigV4), and check that the caller owns the context before you trust it.

## Clean up

The next cell stops every A2A server and checks that its thread has exited. The lab created no AWS resources.

In [ ]:
for server in servers.values():
    server.stop()
a2a_http.close()
assert not any(server.thread.is_alive() for server in servers.values())

In [ ]:
write_validation_artifact("a2a", {
    "peers_called": sorted(called),
    "tools_run": sorted({run["tool"] for run in tool_runs}),
    "same_context_agents": len(first_agents),
    "new_context_agents": len(new_agents),
})

## Takeaways

A2A puts a protocol between the caller and each peer. The caller sees an agent card and replies, and it cannot reach into the peer's tools, model, or memory. Each team can change, redeploy, or rewrite its Agent without the caller changing any code.

`context_id` decides which conversation a message joins, and `agent_factory` gives each context its own Agent. That context is routing only, not authentication, and it lives in the server process.

Check the evidence (who was called, and whether each call succeeded) before you use a decision.

Amazon Bedrock charged for every model call on both sides of each A2A message.

## Where to go next

The [Strands Agent-to-Agent guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/agent-to-agent/) covers the server options, streaming, and `A2AAgent`, a client that treats a remote peer as a Strands Agent. The [A2A specification](https://a2a-protocol.org/) defines agent cards, tasks, and contexts.

## Source and license

The lab uses `A2AServer` from the Strands Agents SDK and `A2AClientToolProvider` from Strands Agents Tools, both licensed under Apache License 2.0.